# 01 — Environment Unit Tests (BESS DA+ID)

Goal: deterministic sanity/unit tests for:
- data integrity (48 SPs, tau 1..48, 30-min cadence)
- timeline advance & rollover
- observation correctness + DA publish gating (no leakage)
- SoC protection (clipping at bounds)
- plan write gating (only after publish)
- reward decomposition consistency

## 0. Setup

### 0.1. Imports + Settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from battery_env import BatteryEnv
import env_config

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

def pretty_passfail(ok: bool) -> str:
    return "PASS" if ok else "FAIL"

S_profit (DA+ID, £): 62.90244249999997
S_carbon (monetised, £): 4.756983517337498


### 0.2. Env factory + deterministic reset

In [2]:
def make_env(seed=123, publish_hour=12, episode_days=10, randomize_init_soc=True):
    env = BatteryEnv(
        config=env_config,
        publish_hour=publish_hour,
        episode_days=episode_days,
        randomize_init_soc=randomize_init_soc,
        seed=seed,
    )
    return env

env = make_env(seed=123, episode_days=10, randomize_init_soc=True)
obs, info = env.reset(seed=123)

print("obs_len:", len(obs))
print("reset info:", info)
print("action_space:", env.action_space)
print("P_max_MW:", env.P_max_MW, "n_power_levels:", env.n_power_levels)
print("SoC bounds:", env.SoC_min, env.SoC_max, "init SoC:", env.soc)

obs_len: 103
reset info: {'start_day': '2023-03-04', 'publish_hour': 12, 'episode_days': 10, 'init_soc': 0.5729407452992574}
action_space: MultiDiscrete([11 11 48])
P_max_MW: 0.18635 n_power_levels: 11
SoC bounds: 0.1 0.9 init SoC: 0.5729407452992574


### 0.3. Generic Stepping + Rollout collector

In [3]:
def step_env(env, action):
    obs, rew, terminated, truncated, info = env.step(action)
    done = bool(terminated or truncated)
    return obs, rew, done, info

def collect_steps(env, n_steps, policy_fn, seed=123):
    obs, info0 = env.reset(seed=seed)
    rows = []
    for t in range(n_steps):
        action = policy_fn(env, obs, t)
        obs, rew, done, info = step_env(env, action)
        row = {"t": t, "reward": float(rew), "done": done}
        # merge info keys if present
        for k, v in info.items():
            row[k] = v
        rows.append(row)
        if done:
            break
    df = pd.DataFrame(rows)
    return df

def zero_action(env):
    # dispatch_idx=middle(≈0 MW), plan_idx=middle, plan_slot=0
    mid = env.n_power_levels // 2
    return np.array([mid, mid, 0], dtype=np.int64)

def random_action(env, rng):
    a = env.action_space.sample()
    return np.array(a, dtype=np.int64)

## 1. Data Integrity Checks

In [4]:
def test_data_integrity(env):
    ok = True
    msgs = []

    # valid_days & day_indices should exist
    if not hasattr(env, "valid_days") or not hasattr(env, "day_indices"):
        return False, ["env missing valid_days/day_indices"]

    if len(env.valid_days) == 0:
        ok = False
        msgs.append("valid_days is empty")

    # check each day_indices has 48 and tau 1..48 and 30-min cadence
    df = env.df
    for d in env.valid_days[:min(50, len(env.valid_days))]:
        idxs = env.day_indices[d]
        if len(idxs) != 48:
            ok = False
            msgs.append(f"{d}: len(idxs)={len(idxs)} != 48")
            continue

        taus = df.loc[idxs, "tau"].to_numpy()
        if set(taus.tolist()) != set(range(1, 49)):
            ok = False
            msgs.append(f"{d}: tau set mismatch (not 1..48)")

        ts = pd.to_datetime(df.loc[idxs, "delivery_ts"])
        deltas = ts.diff().dropna().dt.total_seconds().to_numpy()
        if not np.allclose(deltas, 1800.0):
            ok = False
            msgs.append(f"{d}: delivery_ts not 30-min cadence")

    return ok, msgs

ok, msgs = test_data_integrity(env)
print("Test data_integrity:", pretty_passfail(ok))
if msgs:
    print("\n".join(msgs[:10]))

Test data_integrity: PASS


# 2. Timeline Correctness

## 2.1. Is the step through time done correctly

This test verifies that one environment step corresponds to exactly one 30-minute
interval in the dataset. We check that timestamps advance correctly, tau increments
from 1 to 48 without skips, and the environment remains within the same delivery day
until a full day (48 steps) is completed. This ensures that the temporal structure of
the environment is consistent with the physical market timeline.

In [5]:
def test_timeline_rollover(env, seed=123):
    # step > 48 to force rollover
    def pol(env, obs, t):
        return zero_action(env)

    df = collect_steps(env, n_steps=60, policy_fn=pol, seed=seed)

    ok = True
    msgs = []

    if df.empty:
        return False, ["no steps collected"]

    # tau should increment 1..48 then reset to 1 after day rollover
    taus = df["tau"].to_numpy()
    if not np.all((taus >= 1) & (taus <= 48)):
        ok = False
        msgs.append("tau out of [1,48]")

    # delivery_date should change exactly when tau resets from 48 to 1 (for complete days)
    tau_prev = df["tau"].shift(1)
    date_prev = df["delivery_date"].shift(1)
    rollover_points = df.index[(tau_prev == 48) & (df["tau"] == 1)].tolist()

    for i in rollover_points:
        if df.loc[i, "delivery_date"] == date_prev.loc[i]:
            ok = False
            msgs.append(f"rollover at index {i} but delivery_date did not change")

    return ok, msgs, df

ok, msgs, timeline_df = test_timeline_rollover(env)
print("Test timeline_rollover:", pretty_passfail(ok))
if msgs:
    print("\n".join(msgs[:10]))
timeline_df.head()

Test timeline_rollover: PASS


,t,reward,done,trade_date,delivery_date,tau,decision_ts,delivery_ts,trade_ts,dispatch_idx_exec,dispatch_idx_agent,plan_idx_agent,planned_idx_today,plan_slot_agent,tomorrow_plan_value_written,mef_now,carbon_price_now,idx,P_req_MW,P_planned_MW,P_dev_MW,P_applied_MW,delta_soc,id_price_now,da_price_now,ci_now,Planned_Profit,Intraday_Profit,profit_norm,carbon_penalty_norm,da_available,days_done,soc
0,0,0.0,False,2023-03-03 00:00:00,2023-03-04 00:00:00,1,2023-03-04 00:00:00,2023-03-04 00:00:00,2023-03-03 00:00:00,5,5,5,-1,0,-999,370.0,76.760002,20494,0.0,0.0,0.0,0.0,0.0,134.789993,118.300003,226.0,0.0,0.0,0.0,-0.0,False,0,0.572941
1,1,0.0,False,2023-03-03 00:00:00,2023-03-04 00:00:00,2,2023-03-04 00:30:00,2023-03-04 00:30:00,2023-03-03 00:30:00,5,5,5,-1,0,-999,370.0,76.760002,20495,0.0,0.0,0.0,0.0,0.0,145.970001,140.800003,228.0,0.0,0.0,0.0,-0.0,False,0,0.572941
2,2,0.0,False,2023-03-03 00:00:00,2023-03-04 00:00:00,3,2023-03-04 01:00:00,2023-03-04 01:00:00,2023-03-03 01:00:00,5,5,5,-1,0,-999,370.0,76.760002,20496,0.0,0.0,0.0,0.0,0.0,134.000000,121.000000,230.0,0.0,0.0,0.0,-0.0,False,0,0.572941
3,3,0.0,False,2023-03-03 00:00:00,2023-03-04 00:00:00,4,2023-03-04 01:30:00,2023-03-04 01:30:00,2023-03-03 01:30:00,5,5,5,-1,0,-999,370.0,76.760002,20497,0.0,0.0,0.0,0.0,0.0,126.580002,120.000000,231.0,0.0,0.0,0.0,-0.0,False,0,0.572941
4,4,0.0,False,2023-03-03 00:00:00,2023-03-04 00:00:00,5,2023-03-04 02:00:00,2023-03-04 02:00:00,2023-03-03 02:00:00,5,5,5,-1,0,-999,370.0,76.760002,20498,0.0,0.0,0.0,0.0,0.0,128.889999,123.900002,228.0,0.0,0.0,0.0,-0.0,False,0,0.572941


### 2.2. observation length + DA gating (no leakage)

In [6]:
def test_obs_shape_and_gating(env, seed=123):
    obs, info = env.reset(seed=seed)
    ok = True
    msgs = []

    if len(obs) != 103:
        ok = False
        msgs.append(f"obs length {len(obs)} != 103")

    # Indices based on your obs structure:
    # main = [soc, id_price, ci, tau, p_prev, da_avail, planned_power_now] -> len 7
    # then tomorrow_da (48), tomorrow_ci (48)
    main = obs[:7]
    tomorrow_da = obs[7:7+48]
    tomorrow_ci = obs[7+48:7+48+48]

    da_avail = int(round(float(main[5])))
    if da_avail not in (0,1):
        ok = False
        msgs.append(f"da_avail not binary: {main[5]}")

    if da_avail == 0:
        if not (np.allclose(tomorrow_da, 0.0) and np.allclose(tomorrow_ci, 0.0)):
            ok = False
            msgs.append("DA not available but tomorrow curves are not all zeros (leakage risk)")
    else:
        # if da_avail==1 we expect at least one non-zero typically; allow rare real zero curves but flag if all zeros
        if np.allclose(tomorrow_da, 0.0):
            msgs.append("DA available but tomorrow_da is all zeros (check dataset/tomorrow existence)")

    return ok, msgs

ok, msgs = test_obs_shape_and_gating(env)
print("Test obs_shape_and_gating:", pretty_passfail(ok))
if msgs:
    print("\n".join(msgs))

Test obs_shape_and_gating: PASS


## 3. SoC update sign convention (charging increases SoC, discharging decreases)

In [7]:
def test_soc_update_sign(env, seed=123):
    env.reset(seed=seed)

    ok = True
    msgs = []

    # Put SoC mid-range to avoid clipping
    env.soc = float((env.SoC_min + env.SoC_max) / 2.0)
    env.p_prev = 0.0

    charge_idx = 0
    discharge_idx = env.n_power_levels - 1
    plan_idx = env.n_power_levels // 2
    plan_slot = 0

    # ---- charge ----
    soc0 = env.soc
    _, _, term, trunc, info_c = env.step([charge_idx, plan_idx, plan_slot])
    soc1 = float(info_c.get("soc", env.soc))
    d1 = float(info_c.get("delta_soc", soc1 - soc0))

    if d1 <= 0:
        ok = False
        msgs.append(f"Charging did not increase SoC (delta_soc={d1})")

    # ---- discharge ----
    env.reset(seed=seed)
    env.soc = float((env.SoC_min + env.SoC_max) / 2.0)
    env.p_prev = 0.0

    soc0 = env.soc
    _, _, term, trunc, info_d = env.step([discharge_idx, plan_idx, plan_slot])
    soc1 = float(info_d.get("soc", env.soc))
    d2 = float(info_d.get("delta_soc", soc1 - soc0))

    if d2 >= 0:
        ok = False
        msgs.append(f"Discharging did not decrease SoC (delta_soc={d2})")

    return ok, msgs

ok, msgs = test_soc_update_sign(env)
print("Test soc_update_sign:", pretty_passfail(ok))
if msgs:
    print("\n".join(msgs))

Test soc_update_sign: PASS


In [8]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

eps = 1e-6
N_steps = 5000

obs, info = env.reset(seed=123)
rows = []
fails = []

for t in range(N_steps):
    soc_before = float(env.soc)

    a = env.action_space.sample()          # array([dispatch_idx, plan_idx])
    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    delta = soc_after - soc_before

    P_req = float(env.power_levels[dispatch_idx])
    P_app = float(info.get("P_applied_MW", np.nan))

    # --- sign / direction sanity ---
    ok = True

    # Charge (negative power) => SoC should increase (or stay flat if blocked at max)
    if P_app < -eps and not (soc_after >= soc_before - 1e-8):
        ok = False

    # Discharge (positive power) => SoC should decrease (or stay flat if blocked at min)
    if P_app > eps and not (soc_after <= soc_before + 1e-8):
        ok = False

    # Near-zero power => SoC should barely move
    if abs(P_app) <= eps and not (abs(delta) <= 1e-6):
        ok = False

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": P_req,
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)

    if not ok:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
print("ran steps:", len(df))
print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))
else:
    display(df.head(20))

ran steps: 1440
num fails: 0


,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,5,0,0.00000,0.000000e+00,0.572941,0.572941,0.000000,True,0.000000e+00,2023-02-19 00:00:00,1,False
1,1,0,3,-0.18635,-1.863500e-01,0.572941,0.809261,0.236320,True,-3.045270e-01,2023-02-19 00:30:00,2,False
2,2,5,1,0.00000,0.000000e+00,0.809261,0.809261,0.000000,True,0.000000e+00,2023-02-19 01:00:00,3,False
3,3,7,3,0.07454,7.454000e-02,0.809261,0.712545,-0.096716,True,2.663671e-01,2023-02-19 01:30:00,4,False
4,4,7,9,0.07454,7.454000e-02,0.712545,0.615800,-0.096745,True,2.589303e-01,2023-02-19 02:00:00,5,False
5,5,1,7,-0.14908,-1.490800e-01,0.615800,0.804728,0.188928,True,-1.884197e-01,2023-02-19 02:30:00,6,False
6,6,1,3,-0.14908,-7.513784e-02,0.804728,0.900000,0.095272,True,-8.204872e-02,2023-02-19 03:00:00,7,False
7,7,8,3,0.11181,1.118100e-01,0.900000,0.754489,-0.145511,True,3.467604e-01,2023-02-19 03:30:00,8,False
8,8,1,6,-0.14908,-1.151174e-01,0.754489,0.900000,0.145511,True,-1.188770e-01,2023-02-19 04:00:00,9,False
9,9,4,1,-0.03727,-3.480896e-16,0.900000,0.900000,0.000000,True,-3.684594e-16,2023-02-19 04:30:00,10,False


### 3.3. SoC Sign Consistency

In [17]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
eps = 1e-9

def idx_for_power(env, target_MW: float) -> int:
    return int(np.argmin(np.abs(env.power_levels - target_MW)))

dispatch_charge = idx_for_power(env, -env.P_max_MW)
dispatch_idle   = idx_for_power(env, 0.0)
dispatch_dis    = idx_for_power(env,  env.P_max_MW)

# plan_idx can be anything for this test; keep it simple
plan_idx_const = dispatch_idle

# Build a sequence of 2D actions: [dispatch_idx, plan_idx]
plan = (
    [np.array([dispatch_charge, plan_idx_const, 0], dtype=np.int64)] * 5
    + [np.array([dispatch_idle, plan_idx_const, 0], dtype=np.int64)] * 3
    + [np.array([dispatch_dis, plan_idx_const, 0], dtype=np.int64)] * 5
)

obs, info = env.reset()

rows = []
fails = []

for t, a in enumerate(plan):
    soc_before = float(env.soc)

    obs, reward, terminated, truncated, info = env.step(a)

    soc_after = float(env.soc)
    P_app = float(info["P_applied_MW"])
    delta = soc_after - soc_before

    ok = True
    # Negative power => charge => SoC should go up (or stay ~same if blocked at max)
    if P_app < -eps and soc_after < soc_before - 1e-8:
        ok = False
    # Positive power => discharge => SoC should go down (or stay ~same if blocked at min)
    if P_app > eps and soc_after > soc_before + 1e-8:
        ok = False
    # ~0 power => SoC should barely move
    if abs(P_app) <= eps and abs(delta) > 1e-6:
        ok = False

    dispatch_idx = int(a[0])
    plan_idx = int(a[1])

    row = {
        "t": t,
        "dispatch_idx": dispatch_idx,
        "plan_idx": plan_idx,
        "P_req_MW": float(env.power_levels[dispatch_idx]),
        "P_app_MW": P_app,
        "soc_before": soc_before,
        "soc_after": soc_after,
        "delta_soc": delta,
        "ok_sign": ok,
        "reward": float(reward),
        "trade_ts": info.get("trade_ts", None),
        "tau": info.get("tau", None),
        "da_available": info.get("da_available", None),
    }
    rows.append(row)
    if not ok:
        fails.append(row)

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num sign fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,dispatch_idx,plan_idx,P_req_MW,P_app_MW,soc_before,soc_after,delta_soc,ok_sign,reward,trade_ts,tau,da_available
0,0,0,5,-0.18635,-1.707509e-01,0.685127,0.900000,0.214873,True,-3.430861e-01,2023-04-04 00:00:00,1,False
1,1,0,5,-0.18635,-1.740448e-16,0.900000,0.900000,0.000000,True,-3.633183e-16,2023-04-04 00:30:00,2,False
2,2,0,5,-0.18635,-1.740448e-16,0.900000,0.900000,0.000000,True,-3.519463e-16,2023-04-04 01:00:00,3,False
3,3,0,5,-0.18635,-1.740448e-16,0.900000,0.900000,0.000000,True,-3.544233e-16,2023-04-04 01:30:00,4,False
4,4,0,5,-0.18635,-1.740448e-16,0.900000,0.900000,0.000000,True,-3.733230e-16,2023-04-04 02:00:00,5,False
5,5,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2023-04-04 02:30:00,6,False
6,6,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2023-04-04 03:00:00,7,False
7,7,5,5,0.00000,0.000000e+00,0.900000,0.900000,0.000000,True,0.000000e+00,2023-04-04 03:30:00,8,False
8,8,10,5,0.18635,1.863500e-01,0.900000,0.655846,-0.244154,True,5.588774e-01,2023-04-04 04:00:00,9,False
9,9,10,5,0.18635,1.863500e-01,0.655846,0.411407,-0.244439,True,5.561536e-01,2023-04-04 04:30:00,10,False


num sign fails: 0


## 4. DA publish gating flip (explicit “no cheating”)

In [10]:
def find_publish_flip_in_episode(env, seed=123, max_steps=200):
    """
    Walk forward until we observe da_available flip 0 -> 1.
    Returns dataframe of steps up to flip (or max_steps).
    """
    def pol(env, obs, t):
        return zero_action(env)

    df = collect_steps(env, n_steps=max_steps, policy_fn=pol, seed=seed)
    if "da_available" not in df.columns:
        return df, None

    da = df["da_available"].astype(int).to_numpy()
    flip_idx = None
    for i in range(1, len(da)):
        if da[i-1] == 0 and da[i] == 1:
            flip_idx = i
            break
    return df, flip_idx

df_pub, flip_idx = find_publish_flip_in_episode(env, seed=123, max_steps=200)
print("Found publish flip index:", flip_idx)
df_pub[["t","decision_ts","delivery_ts","delivery_date","tau","da_available"]].head(12)

Found publish flip index: 24


,t,decision_ts,delivery_ts,delivery_date,tau,da_available
0,0,2023-02-20 00:00:00,2023-02-20 00:00:00,2023-02-20 00:00:00,1,False
1,1,2023-02-20 00:30:00,2023-02-20 00:30:00,2023-02-20 00:00:00,2,False
2,2,2023-02-20 01:00:00,2023-02-20 01:00:00,2023-02-20 00:00:00,3,False
3,3,2023-02-20 01:30:00,2023-02-20 01:30:00,2023-02-20 00:00:00,4,False
4,4,2023-02-20 02:00:00,2023-02-20 02:00:00,2023-02-20 00:00:00,5,False
5,5,2023-02-20 02:30:00,2023-02-20 02:30:00,2023-02-20 00:00:00,6,False
6,6,2023-02-20 03:00:00,2023-02-20 03:00:00,2023-02-20 00:00:00,7,False
7,7,2023-02-20 03:30:00,2023-02-20 03:30:00,2023-02-20 00:00:00,8,False
8,8,2023-02-20 04:00:00,2023-02-20 04:00:00,2023-02-20 00:00:00,9,False
9,9,2023-02-20 04:30:00,2023-02-20 04:30:00,2023-02-20 00:00:00,10,False


### 4.2. if flip exists, assert curves are zero before and non-zero after

In [11]:
def test_no_leakage_curves(env, seed=123, max_steps=200):
    env.reset(seed=seed)

    def pol(env, obs, t):
        return zero_action(env)

    df = collect_steps(env, n_steps=max_steps, policy_fn=pol, seed=seed)

    ok = True
    msgs = []

    # Reconstruct observation slices by re-stepping and checking obs directly
    env.reset(seed=seed)
    obs, _ = env.reset(seed=seed)

    # We'll iterate again, checking obs tomorrow curves align with da_available
    for t in range(min(max_steps, 200)):
        obs = env._get_obs()  # uses current state
        main = obs[:7]
        tomorrow_da = obs[7:7+48]
        tomorrow_ci = obs[7+48:7+48+48]
        da_avail = int(round(float(main[5])))

        if da_avail == 0:
            if not (np.allclose(tomorrow_da, 0.0) and np.allclose(tomorrow_ci, 0.0)):
                ok = False
                msgs.append(f"Leakage: da_avail=0 but curves not zero at t={t}")
                break
        else:
            # should correspond to actual data tomorrow if tomorrow exists; at minimum, shouldn't be forced zero always
            if np.allclose(tomorrow_da, 0.0):
                msgs.append(f"da_avail=1 but tomorrow_da all zeros at t={t} (could be missing tomorrow day in dataset)")
                # don't fail hard because dataset boundary can cause zeros

        # step forward
        _, _, done, _info = step_env(env, zero_action(env))
        if done:
            break

    return ok, msgs

ok, msgs = test_no_leakage_curves(env, seed=123)
print("Test no_leakage_curves:", pretty_passfail(ok))
if msgs:
    print("\n".join(msgs[:10]))

Test no_leakage_curves: PASS


In [12]:
def step_env(env, action):
    obs, rew, terminated, truncated, info = env.step(action)
    done = bool(terminated or truncated)
    return obs, rew, done, info

def collect_steps(env, n_steps, policy_fn, seed=123):
    obs, info = env.reset(seed=seed)
    rows = []
    for t in range(n_steps):
        action = policy_fn(env, obs, t)
        obs, rew, done, info = step_env(env, action)
        row = {"t": t, "reward": float(rew), "done": bool(done)}
        if isinstance(info, dict):
            row.update(info)
        rows.append(row)
        if done:
            break
    return pd.DataFrame(rows)

def test_no_leakage_curves(env, seed=123, max_steps=200):
    obs, _ = env.reset(seed=seed)

    ok = True
    msgs = []

    for t in range(min(max_steps, 200)):
        obs = env._get_obs()
        main = obs[:7]
        tomorrow_da = obs[7:7+48]
        tomorrow_ci = obs[7+48:7+48+48]
        da_avail = int(round(float(main[5])))

        if da_avail == 0:
            if not (np.allclose(tomorrow_da, 0.0) and np.allclose(tomorrow_ci, 0.0)):
                ok = False
                msgs.append(f"Leakage: da_avail=0 but curves not zero at t={t}")
                break
        else:
            if np.allclose(tomorrow_da, 0.0):
                msgs.append(f"da_avail=1 but tomorrow_da all zeros at t={t} (maybe boundary/missing tomorrow)")

        _, _, done, _ = step_env(env, zero_action(env))
        if done:
            break

    return ok, msgs

def zero_action(env):
    # pick middle power level (~0 MW if symmetric)
    mid = env.n_power_levels // 2
    return np.array([mid, mid, 0], dtype=np.int64)

ok, msgs = test_no_leakage_curves(env, seed=123)

print("Test no_leakage_curves:", pretty_passfail(ok))

if msgs:
    print("\n".join(msgs))

Test no_leakage_curves: PASS


## 5. Planning write gating (only when DA available)

In [13]:
def test_plan_write_gating(env, seed=123, max_steps=200):
    env.reset(seed=seed)
    ok = True
    msgs = []

    rng = np.random.default_rng(seed)
    plan_slot = 7
    plan_idx = env.n_power_levels - 1  # force something non-default
    dispatch_idx = env.n_power_levels // 2

    wrote_when_unavailable = 0
    wrote_when_available = 0

    for t in range(max_steps):
        # check current availability
        obs = env._get_obs()
        da_avail = int(round(float(obs[5])))  # main[5]
        before = int(env.tomorrow_plan[plan_slot])

        action = np.array([dispatch_idx, plan_idx, plan_slot], dtype=np.int64)
        _obs, _rew, done, info = step_env(env, action)

        after = int(env.tomorrow_plan[plan_slot])
        changed = (after != before)

        if da_avail == 0 and changed:
            wrote_when_unavailable += 1
        if da_avail == 1 and changed:
            wrote_when_available += 1

        if done:
            break

    if wrote_when_unavailable > 0:
        ok = False
        msgs.append(f"tomorrow_plan changed while da_available=0 (count={wrote_when_unavailable})")

    if wrote_when_available == 0:
        msgs.append("tomorrow_plan never changed while da_available=1 (could mean da never became available in this episode)")

    return ok, msgs, {"wrote_when_unavailable": wrote_when_unavailable, "wrote_when_available": wrote_when_available}

ok, msgs, stats = test_plan_write_gating(env, seed=123)
print("Test plan_write_gating:", pretty_passfail(ok), stats)
if msgs:
    print("\n".join(msgs))

Test plan_write_gating: PASS {'wrote_when_unavailable': 0, 'wrote_when_available': 8}


## 5. Plan Rollover Correctness

In [14]:
def test_plan_rollover(env, seed=123):
    env.reset(seed=seed)
    ok = True
    msgs = []

    mid = env.n_power_levels // 2
    target_plan_idx = env.n_power_levels - 1  # e.g., 10 if n_power_levels=11

    # 1) Step until DA becomes available (or give up)
    max_seek = 500
    became_available = False
    for _ in range(max_seek):
        obs = env._get_obs()
        da_avail = int(round(float(obs[5])))
        if da_avail == 1:
            became_available = True
            break
        # step without writing anything meaningful
        env.step([mid, mid, 0])

    if not became_available:
        return False, ["DA never became available; cannot test rollover"], None

    # 2) Write a pattern to tomorrow_plan using actions
    slots = [0, 1, 2, 10, 20, 47]
    for s in slots:
        env.step([mid, target_plan_idx, s])

    written = {s: int(env.tomorrow_plan[s]) for s in slots}

    # 3) Now step to the END of current day WITHOUT overwriting tomorrow_plan
    # Use plan_idx=mid and plan_slot=0 consistently (still writes, but writes mid)
    # Better: only step with da_avail==False would avoid writing, but da_avail is True now.
    # So: we must step with an action that does NOT change any of our tested slots.
    # We'll use a "safe" plan_slot not in slots (e.g., 3).
    safe_slot = 3
    for _ in range(200):
        # find current tau
        idx = int(env.current_day_idxs[env.slot0])
        tau_now = int(env.tau[idx])
        if tau_now == 48:
            break
        env.step([mid, mid, safe_slot])

    # One more step to trigger rollover
    env.step([mid, mid, safe_slot])

    # 4) Check today_plan matches what we wrote
    for s, v in written.items():
        if int(env.today_plan[s]) != v:
            ok = False
            msgs.append(f"rollover mismatch at slot {s}: today_plan={env.today_plan[s]} vs written {v}")

    # 5) Check tomorrow_plan reset
    if not np.all(env.tomorrow_plan == -1):
        ok = False
        msgs.append("tomorrow_plan not reset to -1 after rollover")

    return ok, msgs, written

ok, msgs, written_values = test_plan_rollover(env, seed=123)
print("Test plan_rollover:", pretty_passfail(ok))
print("written_values:", written_values)
if msgs:
    print("\n".join(msgs))

Test plan_rollover: PASS
written_values: {0: 10, 1: 10, 2: 10, 10: 10, 20: 10, 47: 10}


# 6. Reward Decomposition Checks

## 6.1. Profit term sanity with constant price

In [15]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)

# turn off carbon
env.lambda_ci = 0.0

obs, info = env.reset(seed=123)

rows = []
fails = []

N = 80

for t in range(N):
    a = env.action_space.sample()

    obs, reward, terminated, truncated, info = env.step(a)

    P_app = float(info["P_applied_MW"])
    dt = float(env.dt_hours)

    # planned power for *this* slot
    planned_idx = int(info["planned_idx_today"])
    P_plan = float(env.power_levels[planned_idx]) if planned_idx >= 0 else 0.0

    # prices
    da_price = float(info["da_price_now"])
    id_price = float(info["id_price_now"])

    # energies
    E_plan = P_plan * dt
    E_dev  = (P_app - P_plan) * dt

    # raw market revenues (must match env)
    R_DA_calc = E_plan * da_price
    R_ID_calc = E_dev  * id_price
    R_total_calc = R_DA_calc + R_ID_calc

    # env normalises TOTAL profit with profit_scale
    profit_calc = np.tanh(R_total_calc / env.profit_scale)

    ok = np.isclose(float(reward), float(profit_calc), atol=1e-6)

    row = {
        "t": t,
        "tau": info.get("tau"),
        "delivery_ts": info.get("delivery_ts"),
        "P_plan_MW": P_plan,
        "P_app_MW": P_app,
        "da_price": da_price,
        "id_price": id_price,
        "E_plan_MWh": E_plan,
        "E_dev_MWh": E_dev,
        "R_DA_calc": R_DA_calc,
        "R_ID_calc": R_ID_calc,
        "R_total_calc": R_total_calc,
        "reward_env": float(reward),
        "profit_calc": float(profit_calc),
        "ok_profit_only": bool(ok),
    }
    rows.append(row)

    if not ok:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,tau,delivery_ts,P_plan_MW,P_app_MW,da_price,id_price,E_plan_MWh,E_dev_MWh,R_DA_calc,R_ID_calc,R_total_calc,reward_env,profit_calc,ok_profit_only
0,0,1,2023-02-20 00:00:00,0.00000,0.037270,65.000000,86.570000,0.000000,0.018635,0.000000,1.613232,1.613232,0.025641,0.025641,True
1,1,2,2023-02-20 00:30:00,0.00000,0.111810,86.199997,109.110001,0.000000,0.055905,0.000000,6.099795,6.099795,0.096669,0.096669,True
2,2,3,2023-02-20 01:00:00,0.00000,-0.037270,71.699997,108.730003,0.000000,-0.018635,0.000000,-2.026184,-2.026184,-0.032200,-0.032200,True
3,3,4,2023-02-20 01:30:00,0.00000,-0.037270,68.000000,99.629997,0.000000,-0.018635,0.000000,-1.856605,-1.856605,-0.029507,-0.029507,True
4,4,5,2023-02-20 02:00:00,0.00000,0.037270,62.200001,87.040001,0.000000,0.018635,0.000000,1.621990,1.621990,0.025780,0.025780,True
5,5,6,2023-02-20 02:30:00,0.00000,0.074540,60.000000,76.690002,0.000000,0.037270,0.000000,2.858236,2.858236,0.045408,0.045408,True
6,6,7,2023-02-20 03:00:00,0.00000,-0.037270,58.099998,55.590000,0.000000,-0.018635,0.000000,-1.035920,-1.035920,-0.016467,-0.016467,True
7,7,8,2023-02-20 03:30:00,0.00000,0.037270,47.900002,47.770000,0.000000,0.018635,0.000000,0.890194,0.890194,0.014151,0.014151,True
8,8,9,2023-02-20 04:00:00,0.00000,0.149080,47.799999,46.330002,0.000000,0.074540,0.000000,3.453438,3.453438,0.054846,0.054846,True
9,9,10,2023-02-20 04:30:00,0.00000,-0.149080,50.000000,51.279999,0.000000,-0.074540,0.000000,-3.822411,-3.822411,-0.060693,-0.060693,True


num fails: 0


## 7.2. Reward Check including Carbon 

In [16]:
import numpy as np
import pandas as pd
from battery_env import BatteryEnv
import env_config

env = BatteryEnv(env_config)
obs, info = env.reset(seed=123)

rows = []
fails = []

N = 80

for t in range(N):
    a = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(a)

    P_app = float(info["P_applied_MW"])
    dt = float(env.dt_hours)

    planned_idx = int(info["planned_idx_today"])
    P_plan = float(env.power_levels[planned_idx]) if planned_idx >= 0 else 0.0

    da_price = float(info["da_price_now"])
    id_price = float(info["id_price_now"])
    ci_now   = float(info["ci_now"])

    # IMPORTANT: mef + carbon price are NOT in info in your current env output,
    # so read them from env arrays using current step index.
    # We reconstruct idx the same way env does: current slot row index.
    idx = int(env.current_day_idxs[env.slot0 - 1])  # -1 because env already advanced time after step()
    mef_now = float(env.mef[idx])
    carbon_price_now = float(env.carbon_price[idx])

    # energies (MWh)
    E_plan = P_plan * dt
    E_dev  = (P_app - P_plan) * dt
    E_act  = P_app * dt

    # ---------- PROFIT (matches env) ----------
    R_DA_calc = E_plan * da_price
    R_ID_calc = E_dev  * id_price
    R_total_calc = R_DA_calc + R_ID_calc

    profit_calc = np.tanh(R_total_calc / env.profit_scale)

    # ---------- CARBON (matches env exactly) ----------
    # env:
    #   E_import_kWh = max(-E_act_MWh, 0) * 1000
    #   E_export_kWh = max( E_act_MWh, 0) * 1000
    #   net_tCO2 = (E_import_kWh*ci_now - E_export_kWh*mef_now) / 1e6
    #   carbon_cashflow_gbp = carbon_price_now * net_tCO2
    #   carbon_norm = tanh(carbon_cashflow_gbp / carbon_scale)
    #   reward = profit_norm - lambda_ci * carbon_norm

    E_import_kWh = max(-E_act, 0.0) * 1000.0
    E_export_kWh = max( E_act, 0.0) * 1000.0

    net_tCO2 = (E_import_kWh * ci_now - E_export_kWh * mef_now) / 1e6
    carbon_cashflow_gbp = carbon_price_now * net_tCO2
    carbon_calc = np.tanh(carbon_cashflow_gbp / env.carbon_scale)

    reward_calc = profit_calc - env.lambda_ci * carbon_calc

    ok = np.isclose(float(reward), float(reward_calc), atol=1e-6)

    row = {
        "t": t,
        "tau": info.get("tau"),
        "delivery_ts": info.get("delivery_ts"),
        "P_plan_MW": P_plan,
        "P_app_MW": P_app,
        "da_price": da_price,
        "id_price": id_price,
        "ci_now": ci_now,
        "mef_now": mef_now,
        "uka_gbp_tco2": carbon_price_now,
        "E_plan_MWh": E_plan,
        "E_dev_MWh": E_dev,
        "E_act_MWh": E_act,
        "R_total_calc": R_total_calc,
        "profit_env": float(info["profit_norm"]),
        "profit_calc": float(profit_calc),
        "carbon_env": float(info["carbon_penalty_norm"]),
        "carbon_calc": float(carbon_calc),
        "reward_env": float(reward),
        "reward_calc": float(reward_calc),
        "ok_reward": bool(ok),
        "mef_now": float(info["mef_now"]),
        "carbon_price_now":float(info["carbon_price_now"]),
    }
    rows.append(row)

    if not ok:
        fails.append(row)
        break

    if terminated or truncated:
        break

df = pd.DataFrame(rows)
display(df)

print("num fails:", len(fails))
if fails:
    display(pd.DataFrame(fails))

,t,tau,delivery_ts,P_plan_MW,P_app_MW,da_price,id_price,ci_now,mef_now,uka_gbp_tco2,E_plan_MWh,E_dev_MWh,E_act_MWh,R_total_calc,profit_env,profit_calc,carbon_env,carbon_calc,reward_env,reward_calc,ok_reward,carbon_price_now
0,0,1,2023-02-20 00:00:00,0.00000,-3.727000e-02,65.000000,86.570000,97.0,370.0,80.949997,0.000000,-1.863500e-02,-1.863500e-02,-1.613232e+00,-2.564095e-02,-2.564095e-02,3.075030e-02,3.075030e-02,-5.331622e-02,-5.331622e-02,True,80.949997
1,1,2,2023-02-20 00:30:00,0.00000,1.490800e-01,86.199997,109.110001,102.0,370.0,80.949997,0.000000,7.454000e-02,7.454000e-02,8.133059e+00,1.285807e-01,1.285807e-01,-4.376560e-01,-4.376560e-01,5.224711e-01,5.224711e-01,True,80.949997
2,2,3,2023-02-20 01:00:00,0.00000,-7.454000e-02,71.699997,108.730003,101.0,370.0,80.949997,0.000000,-3.727000e-02,-3.727000e-02,-4.052367e+00,-6.433407e-02,-6.433407e-02,6.396943e-02,6.396943e-02,-1.219066e-01,-1.219066e-01,True,80.949997
3,3,4,2023-02-20 01:30:00,0.00000,1.863500e-01,68.000000,99.629997,95.0,370.0,80.949997,0.000000,9.317500e-02,9.317500e-02,9.283025e+00,1.465160e-01,1.465160e-01,-5.274890e-01,-5.274890e-01,6.212561e-01,6.212561e-01,True,80.949997
4,4,5,2023-02-20 02:00:00,0.00000,-7.454000e-02,62.200001,87.040001,92.0,370.0,80.949997,0.000000,-3.727000e-02,-3.727000e-02,-3.243981e+00,-5.152595e-02,-5.152595e-02,5.828273e-02,5.828273e-02,-1.039804e-01,-1.039804e-01,True,80.949997
5,5,6,2023-02-20 02:30:00,0.00000,7.454000e-02,60.000000,76.690002,86.0,370.0,80.949997,0.000000,3.727000e-02,3.727000e-02,2.858236e+00,4.540795e-02,4.540795e-02,-2.304493e-01,-2.304493e-01,2.528123e-01,2.528123e-01,True,80.949997
6,6,7,2023-02-20 03:00:00,0.00000,3.727000e-02,58.099998,55.590000,85.0,370.0,80.949997,0.000000,1.863500e-02,1.863500e-02,1.035920e+00,1.646718e-02,1.646718e-02,-1.167965e-01,-1.167965e-01,1.215840e-01,1.215840e-01,True,80.949997
7,7,8,2023-02-20 03:30:00,0.00000,-3.727000e-02,47.900002,47.770000,85.0,370.0,80.949997,0.000000,-1.863500e-02,-1.863500e-02,-8.901939e-01,-1.415103e-02,-1.415103e-02,2.694811e-02,2.694811e-02,-3.840433e-02,-3.840433e-02,True,80.949997
8,8,9,2023-02-20 04:00:00,0.00000,3.727000e-02,47.799999,46.330002,87.0,370.0,80.949997,0.000000,1.863500e-02,1.863500e-02,8.633595e-01,1.372451e-02,1.372451e-02,-1.167965e-01,-1.167965e-01,1.188413e-01,1.188413e-01,True,80.949997
9,9,10,2023-02-20 04:30:00,0.00000,-3.727000e-02,50.000000,51.279999,85.0,370.0,80.949997,0.000000,-1.863500e-02,-1.863500e-02,-9.556027e-01,-1.519065e-02,-1.519065e-02,2.694811e-02,2.694811e-02,-3.944395e-02,-3.944395e-02,True,80.949997


num fails: 0
